# 🤖 LLM Fine-Tuning — Your Dataset Edition
> **Datasets:** `combined_emotion.csv` · `combined_sentiment_data.csv`

| Dataset | Rows | Task | Labels |
|---------|------|------|--------|
| combined_emotion.csv | 422,746 | Emotion Classification | joy, sad, anger, fear, love, suprise |
| combined_sentiment_data.csv | 3,309 | Sentiment Analysis | positive, negative |

---
## 📚 Notebook Structure
- **Stage 1** — Data Exploration & Visualization
- **Stage 2** — Sentiment Fine-Tuning (DistilBERT on sentiment data)
- **Stage 3** — Emotion Fine-Tuning (BERT on emotion data)
- **Stage 4** — PEFT / LoRA Fine-Tuning (Efficient training)
- **Stage 5** — Inference & Evaluation


In [ ]:
# ── Install all required libraries ───────────────────────────────────────────
!pip install transformers datasets evaluate accelerate peft bitsandbytes torch -q

---
# 📊 STAGE 1: Data Exploration & Visualization


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Load datasets ─────────────────────────────────────────────────────────────
emotion_df   = pd.read_csv('combined_emotion.csv')
sentiment_df = pd.read_csv('combined_sentiment_data.csv')

print('=== Emotion Dataset ===')
print(f'Shape: {emotion_df.shape}')
print(emotion_df.head())

print('\n=== Sentiment Dataset ===')
print(f'Shape: {sentiment_df.shape}')
print(sentiment_df.head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Emotion distribution
emotion_counts = emotion_df['emotion'].value_counts()
colors_e = ['#FF6B6B','#4ECDC4','#45B7D1','#96CEB4','#FFEAA7','#DDA0DD']
axes[0].bar(emotion_counts.index, emotion_counts.values, color=colors_e)
axes[0].set_title('Emotion Dataset — Label Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Emotion')
axes[0].set_ylabel('Count')
for i, v in enumerate(emotion_counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontsize=9)

# Sentiment distribution
sentiment_counts = sentiment_df['sentiment'].value_counts()
colors_s = ['#2ECC71','#E74C3C']
axes[1].bar(sentiment_counts.index, sentiment_counts.values, color=colors_s, width=0.4)
axes[1].set_title('Sentiment Dataset — Label Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Sentiment')
axes[1].set_ylabel('Count')
for i, v in enumerate(sentiment_counts.values):
    axes[1].text(i, v + 10, f'{v:,}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('dataset_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Text length analysis ──────────────────────────────────────────────────────
emotion_df['text_len']   = emotion_df['sentence'].str.split().str.len()
sentiment_df['text_len'] = sentiment_df['sentence'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(emotion_df['text_len'], bins=50, color='#45B7D1', edgecolor='white')
axes[0].set_title('Emotion — Text Length Distribution')
axes[0].set_xlabel('Word Count')
axes[0].axvline(emotion_df['text_len'].median(), color='red', linestyle='--', label=f'Median: {emotion_df["text_len"].median():.0f}')
axes[0].legend()

axes[1].hist(sentiment_df['text_len'], bins=30, color='#FF6B6B', edgecolor='white')
axes[1].set_title('Sentiment — Text Length Distribution')
axes[1].set_xlabel('Word Count')
axes[1].axvline(sentiment_df['text_len'].median(), color='blue', linestyle='--', label=f'Median: {sentiment_df["text_len"].median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Emotion  — avg words:', round(emotion_df['text_len'].mean(), 1),
      '| max:', emotion_df['text_len'].max())
print('Sentiment — avg words:', round(sentiment_df['text_len'].mean(), 1),
      '| max:', sentiment_df['text_len'].max())

---
# ⚙️ STAGE 2: Sentiment Fine-Tuning (DistilBERT)
### Dataset: `combined_sentiment_data.csv` — 3,309 rows | positive / negative


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import evaluate
import numpy as np
from sklearn.model_selection import train_test_split

# ── Label encoding ────────────────────────────────────────────────────────────
SENTIMENT_LABELS = {'negative': 0, 'positive': 1}
sentiment_df['label'] = sentiment_df['sentiment'].map(SENTIMENT_LABELS)

train_s, test_s = train_test_split(sentiment_df, test_size=0.2, random_state=42,
                                    stratify=sentiment_df['label'])
print(f'Train: {len(train_s)} | Test: {len(test_s)}')
print('Label distribution (train):\n', train_s['label'].value_counts())

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(examples['sentence'], truncation=True,
                     padding='max_length', max_length=128)

train_ds = Dataset.from_pandas(train_s[['sentence','label']].reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_s[['sentence','label']].reset_index(drop=True))

train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

train_ds.set_format('torch', columns=['input_ids','attention_mask','label'])
test_ds.set_format('torch',  columns=['input_ids','attention_mask','label'])

print('Tokenization done!')
print('Sample token IDs:', train_ds[0]['input_ids'][:10])

In [ ]:
id2label = {0: 'negative', 1: 'positive'}
label2id = {'negative': 0, 'positive': 1}

sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

accuracy = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy.compute(predictions=preds, references=labels)
    f1  = f1_metric.compute(predictions=preds, references=labels, average='weighted')
    return {**acc, **f1}

training_args = TrainingArguments(
    output_dir='./sentiment_model',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,
    weight_decay=0.01,
    learning_rate=2e-5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=20,
    report_to='none',
)

trainer_s = Trainer(
    model=sentiment_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

print('Training sentiment model...')
trainer_s.train()

In [ ]:
results_s = trainer_s.evaluate()
print('\n✅ Sentiment Model Results:')
for k, v in results_s.items():
    print(f'  {k}: {v:.4f}')

trainer_s.save_model('./sentiment_model_final')
print('\nModel saved to ./sentiment_model_final')

---
# 🎭 STAGE 3: Emotion Fine-Tuning (BERT)
### Dataset: `combined_emotion.csv` — 422,746 rows | 6 emotions


In [ ]:
from sklearn.model_selection import train_test_split

# ── Label encoding ────────────────────────────────────────────────────────────
EMOTION_LABELS = {'joy': 0, 'sad': 1, 'anger': 2, 'fear': 3, 'love': 4, 'suprise': 5}
ID2EMOTION     = {v: k for k, v in EMOTION_LABELS.items()}

emotion_df['label'] = emotion_df['emotion'].map(EMOTION_LABELS)

# Use 50k subset for faster training (remove .sample() for full dataset)
emotion_sample = emotion_df.sample(n=50000, random_state=42)

train_e, test_e = train_test_split(emotion_sample, test_size=0.15,
                                    random_state=42, stratify=emotion_sample['label'])

print(f'Train: {len(train_e)} | Test: {len(test_e)}')
print('Label distribution (train):\n', train_e['emotion'].value_counts())

In [ ]:
EMOTION_MODEL = 'bert-base-uncased'
emotion_tokenizer = AutoTokenizer.from_pretrained(EMOTION_MODEL)

def tokenize_emotion(examples):
    return emotion_tokenizer(examples['sentence'], truncation=True,
                              padding='max_length', max_length=128)

train_eds = Dataset.from_pandas(train_e[['sentence','label']].reset_index(drop=True))
test_eds  = Dataset.from_pandas(test_e[['sentence','label']].reset_index(drop=True))

train_eds = train_eds.map(tokenize_emotion, batched=True)
test_eds  = test_eds.map(tokenize_emotion, batched=True)

train_eds.set_format('torch', columns=['input_ids','attention_mask','label'])
test_eds.set_format('torch',  columns=['input_ids','attention_mask','label'])

print('Tokenization done!')

In [ ]:
emotion_model = AutoModelForSequenceClassification.from_pretrained(
    EMOTION_MODEL,
    num_labels=6,
    id2label=ID2EMOTION,
    label2id=EMOTION_LABELS,
)

def compute_metrics_multi(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy.compute(predictions=preds, references=labels)
    f1  = f1_metric.compute(predictions=preds, references=labels, average='weighted')
    return {**acc, **f1}

emotion_args = TrainingArguments(
    output_dir='./emotion_model',
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_ratio=0.1,
    weight_decay=0.01,
    learning_rate=2e-5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    fp16=torch.cuda.is_available(),   # use fp16 on GPU
    logging_steps=100,
    report_to='none',
)

trainer_e = Trainer(
    model=emotion_model,
    args=emotion_args,
    train_dataset=train_eds,
    eval_dataset=test_eds,
    compute_metrics=compute_metrics_multi,
)

print('Training emotion model...')
trainer_e.train()

In [ ]:
results_e = trainer_e.evaluate()
print('\n✅ Emotion Model Results:')
for k, v in results_e.items():
    print(f'  {k}: {v:.4f}')

trainer_e.save_model('./emotion_model_final')
print('\nModel saved to ./emotion_model_final')

---
# 🚀 STAGE 4: PEFT / LoRA Fine-Tuning (Efficient Training)
### Fine-tune RoBERTa on Emotion dataset using LoRA — uses only ~0.5% trainable params


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForSequenceClassification, AutoTokenizer

LORA_MODEL = 'roberta-base'
lora_tokenizer = AutoTokenizer.from_pretrained(LORA_MODEL)

def tokenize_lora(examples):
    return lora_tokenizer(examples['sentence'], truncation=True,
                           padding='max_length', max_length=128)

train_lora = Dataset.from_pandas(train_e[['sentence','label']].reset_index(drop=True))
test_lora  = Dataset.from_pandas(test_e[['sentence','label']].reset_index(drop=True))

train_lora = train_lora.map(tokenize_lora, batched=True)
test_lora  = test_lora.map(tokenize_lora, batched=True)

train_lora.set_format('torch', columns=['input_ids','attention_mask','label'])
test_lora.set_format('torch',  columns=['input_ids','attention_mask','label'])

# Load base model
base_model = AutoModelForSequenceClassification.from_pretrained(
    LORA_MODEL,
    num_labels=6,
    id2label=ID2EMOTION,
    label2id=EMOTION_LABELS,
)

# Apply LoRA config
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,                          # rank
    lora_alpha=16,                # scaling factor
    target_modules=['query','value'],  # apply to attention layers
    lora_dropout=0.1,
    bias='none',
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()
# Expected: trainable params: ~300K | all params: ~125M | ~0.24%

In [ ]:
lora_args = TrainingArguments(
    output_dir='./lora_emotion_model',
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_ratio=0.1,
    learning_rate=3e-4,           # higher LR works better with LoRA
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to='none',
)

trainer_lora = Trainer(
    model=lora_model,
    args=lora_args,
    train_dataset=train_lora,
    eval_dataset=test_lora,
    compute_metrics=compute_metrics_multi,
)

print('Training LoRA model...')
trainer_lora.train()

In [ ]:
results_lora = trainer_lora.evaluate()
print('\n✅ LoRA Emotion Model Results:')
for k, v in results_lora.items():
    print(f'  {k}: {v:.4f}')

trainer_lora.save_model('./lora_emotion_final')
print('LoRA adapter saved!')

---
# 🎯 STAGE 5: Inference & Evaluation


In [ ]:
# ── Sentiment Inference ───────────────────────────────────────────────────────
from transformers import pipeline

sentiment_pipe = pipeline(
    'text-classification',
    model='./sentiment_model_final',
    tokenizer='distilbert-base-uncased',
)

test_texts_s = [
    'This product is absolutely amazing, I love it!',
    'Worst purchase I have ever made. Terrible quality.',
    'It is okay, nothing special about it.',
    'Exceeded all my expectations, highly recommended!',
]

print('=== Sentiment Predictions ===')
for text in test_texts_s:
    result = sentiment_pipe(text)[0]
    emoji = '😊' if result['label'] == 'positive' else '😞'
    print(f'{emoji} [{result["label"].upper()} {result["score"]:.2%}] {text[:60]}')

In [ ]:
# ── Emotion Inference ─────────────────────────────────────────────────────────
emotion_pipe = pipeline(
    'text-classification',
    model='./emotion_model_final',
    tokenizer='bert-base-uncased',
)

test_texts_e = [
    'I am so happy today, everything is going great!',
    'I feel really scared and helpless right now.',
    'This makes me so angry, I cannot believe it!',
    'I miss you so much, my heart aches.',
    'I am deeply in love with this moment.',
    'Wow, I did not expect that at all!',
]

EMOTION_EMOJI = {'joy':'😄','sad':'😢','anger':'😠','fear':'😨','love':'❤️','suprise':'😲'}

print('=== Emotion Predictions ===')
for text in test_texts_e:
    result = emotion_pipe(text)[0]
    label  = result['label']
    emoji  = EMOTION_EMOJI.get(label, '🎭')
    print(f'{emoji} [{label.upper()} {result["score"]:.2%}] {text[:60]}')

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Get predictions on test set
preds_e   = trainer_e.predict(test_eds)
y_pred    = np.argmax(preds_e.predictions, axis=-1)
y_true    = preds_e.label_ids
cm        = confusion_matrix(y_true, y_pred)
labels    = ['joy','sad','anger','fear','love','suprise']

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True', fontsize=11)
ax.set_title('Emotion Model — Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=labels))

In [ ]:
# ── Final Model Comparison ────────────────────────────────────────────────────
comparison = {
    'Model'           : ['DistilBERT (Sentiment)', 'BERT (Emotion)', 'RoBERTa+LoRA (Emotion)'],
    'Dataset'         : ['combined_sentiment_data', 'combined_emotion', 'combined_emotion'],
    'Train Samples'   : [len(train_s), len(train_e), len(train_e)],
    'Labels'          : [2, 6, 6],
    'Accuracy'        : [
        results_s.get('eval_accuracy', 0),
        results_e.get('eval_accuracy', 0),
        results_lora.get('eval_accuracy', 0),
    ],
    'F1 Score'        : [
        results_s.get('eval_f1', 0),
        results_e.get('eval_f1', 0),
        results_lora.get('eval_f1', 0),
    ],
    'Trainable Params': ['66M (100%)', '110M (100%)', '~300K (0.24%)']
}

cmp_df = pd.DataFrame(comparison)
print('\n=== Final Model Comparison ===')
print(cmp_df.to_string(index=False))

---
## ✅ Summary

| Model | Task | Dataset | Technique |
|-------|------|---------|----------|
| DistilBERT | Sentiment (pos/neg) | combined_sentiment_data.csv | Full fine-tune |
| BERT | Emotion (6 classes) | combined_emotion.csv | Full fine-tune |
| RoBERTa + LoRA | Emotion (6 classes) | combined_emotion.csv | PEFT / LoRA |

### 🚀 Next Steps
- Push models to **Hugging Face Hub** with `model.push_to_hub('your-model-name')`
- Build a **Gradio demo** for live inference
- Try **QLoRA** on a 7B model (Mistral/Llama) for generative emotion responses
- Add **multilingual support** (Bangla emotion detection)
